# DiffusionGemma 26B-A4B-it · Kaggle T4×2

目标：在 **2× Tesla T4** 上真正跑通 `google/diffusiongemma`。

优先把 MoE 专家打成 NF4、注意力保持 FP16（避免 RoPE `view` 崩）。
若 4-bit generate 仍失败，回退到已验证的 FP16 + CPU offload。

In [ ]:
import os, subprocess, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "accelerate", "bitsandbytes", "kagglehub",
])
print("pip ok")

In [ ]:
import json, time, traceback, gc
from pathlib import Path

import torch
import transformers
from transformers import AutoProcessor, BitsAndBytesConfig, DiffusionGemmaForBlockDiffusion

print("torch", torch.__version__, "cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("cuda_available", torch.cuda.is_available(), "gpu_count", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"gpu{i}", p.name, f"{p.total_memory/1024**3:.2f} GiB")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "need T4 x2"

def patch_rope():
    import transformers.models.gemma4.modeling_gemma4 as g4
    orig = g4.apply_rotary_pos_emb
    if getattr(orig, "_dg_patched", False):
        return
    def wrapped(x, cos, sin, unsqueeze_dim=1):
        cos = cos.to(device=x.device, dtype=x.dtype)
        sin = sin.to(device=x.device, dtype=x.dtype)
        return orig(x, cos, sin, unsqueeze_dim=unsqueeze_dim)
    wrapped._dg_patched = True
    g4.apply_rotary_pos_emb = wrapped
    try:
        import transformers.models.diffusion_gemma.modeling_diffusion_gemma as dg
        dg.apply_rotary_pos_emb = wrapped
    except Exception:
        pass
    print("rope patched")

patch_rope()

In [ ]:
CANDIDATES = [
    "/kaggle/input/models/google/diffusiongemma/transformers/diffusiongemma-26b-a4b-it/1",
    "/kaggle/input/diffusiongemma/transformers/diffusiongemma-26b-a4b-it/1",
    "/kaggle/input/diffusiongemma/transformers/diffusiongemma-26b-a4b-it",
    "/kaggle/input/google/diffusiongemma/transformers/diffusiongemma-26b-a4b-it/1",
]
MODEL_ID = next((p for p in CANDIDATES if os.path.exists(p)), None)
if MODEL_ID is None:
    import kagglehub
    MODEL_ID = kagglehub.model_download("google/diffusiongemma/transformers/diffusiongemma-26b-a4b-it")
print("MODEL_ID", MODEL_ID)

SKIP = [
    "embed_tokens", "lm_head", "vision_tower", "multi_modal_projector",
    "q_proj", "k_proj", "v_proj", "o_proj", "q_norm", "k_norm", "v_norm",
    "rotary_emb", "input_layernorm", "post_attention_layernorm",
]

def gpu_only_mem():
    return {i: "14.4GiB" for i in range(torch.cuda.device_count())}

def gpu_cpu_mem():
    mem = gpu_only_mem()
    mem["cpu"] = "28GiB"
    return mem

def nf4_cfg(skip=True, offload=False):
    kw = dict(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    if skip:
        kw["llm_int8_skip_modules"] = SKIP
    if offload:
        kw["llm_int8_enable_fp32_cpu_offload"] = True
    return BitsAndBytesConfig(**kw)

STRATEGIES = [
    ("nf4-moe-gpu",
     dict(quantization_config=nf4_cfg(skip=True, offload=False), device_map="auto", max_memory=gpu_only_mem())),
    ("nf4-full-gpu",
     dict(quantization_config=nf4_cfg(skip=False, offload=False), device_map="auto", max_memory=gpu_only_mem())),
    ("nf4-moe-offload",
     dict(quantization_config=nf4_cfg(skip=True, offload=True), device_map="auto", max_memory=gpu_cpu_mem())),
    ("fp16-offload",
     dict(torch_dtype=torch.float16, device_map="auto", max_memory=gpu_cpu_mem(), low_cpu_mem_usage=True)),
]

In [ ]:
def device_summary(model):
    counts = {}
    for _, p in model.named_parameters():
        dev = str(p.device)
        counts[dev] = counts.get(dev, 0) + p.numel()
    out = {k: round(v / 1e9, 3) for k, v in counts.items()}
    print("param_billions_by_device", out, flush=True)
    return out

def mem_now():
    d = {}
    for i in range(torch.cuda.device_count()):
        d[f"gpu{i}_alloc_gib"] = round(torch.cuda.memory_allocated(i) / 1024**3, 3)
        d[f"gpu{i}_reserved_gib"] = round(torch.cuda.memory_reserved(i) / 1024**3, 3)
    return d

def drop(model=None):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()

def decode_text(processor, output):
    raw = processor.decode(output[0], skip_special_tokens=False)
    if isinstance(raw, list):
        raw = raw[0]
    return str(raw).replace(chr(60) + "pad>", "").strip()

def generate_once(model, processor, messages, max_new_tokens=256):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    )
    inputs = {k: v.to("cuda:0") if hasattr(v, "to") else v for k, v in inputs.items()}
    print("input_ids", tuple(inputs["input_ids"].shape), flush=True)
    t0 = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    elapsed = time.perf_counter() - t0
    text = decode_text(processor, output)
    return text, elapsed

processor = AutoProcessor.from_pretrained(MODEL_ID)
messages = [{"role": "user", "content": "Explain in 4 short sentences what discrete diffusion language models do."}]
load_errors, gen_errors = [], []
model = text = load_name = None
elapsed = None
param_map = {}

for name, kwargs in STRATEGIES:
    print("\n=== load", name, "===", flush=True)
    drop(model)
    t0 = time.perf_counter()
    try:
        model = DiffusionGemmaForBlockDiffusion.from_pretrained(MODEL_ID, **kwargs)
        print("loaded", name, "in", round(time.perf_counter() - t0, 1), "s", flush=True)
        param_map = device_summary(model)
        print("mem", mem_now(), flush=True)
    except Exception as e:
        msg = f"{name} load: {type(e).__name__}: {e}"
        load_errors.append(msg)
        print(msg, flush=True)
        traceback.print_exc()
        model = None
        continue
    try:
        text, elapsed = generate_once(model, processor, messages)
        load_name = name
        print("generate_s", round(elapsed, 2), flush=True)
        print(text, flush=True)
        break
    except Exception as e:
        msg = f"{name} generate: {type(e).__name__}: {e}"
        gen_errors.append(msg)
        print(msg, flush=True)
        traceback.print_exc()
        drop(model)
        model = None

assert text is not None, "no strategy generated\n" + "\n".join(load_errors + gen_errors)
print("USING", load_name)

In [ ]:
results = {
    "ok": True,
    "model_id": MODEL_ID,
    "load_strategy": load_name,
    "load_errors": load_errors,
    "gen_errors": gen_errors,
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "generate_s": round(elapsed, 3),
    "prompt": messages[0]["content"],
    "text": text,
    "param_billions_by_device": param_map,
    "memory": mem_now(),
}
Path("/kaggle/working/results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False))
Path("/kaggle/working/generation.txt").write_text(str(text))
print(json.dumps({k: results[k] for k in ("ok", "load_strategy", "gpu_count", "gpus", "generate_s", "memory", "param_billions_by_device")}, indent=2))
print("WROTE results.json")